# PROJET MODELISATION GEOMETRIQUE

## SUJET

**Surfacique : nuage de points**

Pour un nuage de points donné, nous cherchons à retrouver une normale et un rayon en chaque point. Par la suite, nous chercherons à trouver un point sur la surface définie par le nuage de points.

In [262]:
# Imports
import numpy as np
import plotly.graph_objects as go
import open3d as o3d

from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from utils import *

# Paramètres
NUM_POINTS = 100                    # Nombre de points
NUM_NEIGHBORS = 6                   # Nombre de voisin par point
k = 2                               # Pour le calcul de k_near_neighbors

# Type de génération
SIZE = 5
GENERATION_TYPE = 'surface'

### Générer le nuage de points

In [263]:

def generate_points(size, num_points, distribution):
    """
    Générer des points en 3D en fonction de la taille spécifiée, du nombre de points et de la distribution.
    Paramètres :
    - size (float) : La taille de l'espace dans lequel les points seront générés.
    - num_points (int) : Le nombre de points à générer.
    - distribution (str) : Le type de distribution des points.
    Renvoie :
    - points (ndarray) : Un tableau de forme (num_points, 3) contenant les points en 3D générés.
    - center (list) : Les coordonnées du centre des points générés pour orienter les normales
    """
    center = [0, 0, 0]

    if distribution == 'surface':
        edge = int(np.ceil(np.sqrt(num_points)))
        x = np.linspace(-size, size, edge)
        y = np.linspace(-size, size, edge)
        x, y = np.meshgrid(x, y)
        z = x**2 - y**2
        z *= size/np.max(z)
        points = np.column_stack((x.flatten(), y.flatten(), z.flatten()))[:NUM_POINTS]
        center = [0, 0, -size]

    elif distribution == 'regular_sphere':
        indices = np.arange(0, num_points, dtype=float) + 0.5
        phi = np.arccos(1 - 2 * indices/num_points)
        theta = np.pi * (1 + 5**0.5) * indices
        x = size * np.cos(theta) * np.sin(phi)
        y = size * np.sin(theta) * np.sin(phi)
        z = size * np.cos(phi)
        points = np.column_stack((x, y, z))

    elif distribution == 'random_sphere':
        theta = 2 * np.pi * np.random.rand(num_points)
        phi = np.arccos(2 * np.random.rand(num_points) - 1)
        x = size * np.sin(phi) * np.cos(theta)
        y = size * np.sin(phi) * np.sin(theta)
        z = size * np.cos(phi)
        points = np.column_stack((x, y, z))

    else:
        fichier_xyz = "models3D/" + distribution + ".xyz"
        points = np.loadtxt(fichier_xyz)
        echant = np.linspace(0, points.shape[0] - 1, NUM_POINTS, dtype=int)
        points = points[echant]
        points[:, 1] = -points[:, 1]
        points *= size/np.max(points)
        center = [np.mean(points[:, 0]), np.mean(points[:, 1]), np.mean(points[:, 2])]

    return points, center


points, center = generate_points(size=SIZE, num_points=NUM_POINTS, distribution=GENERATION_TYPE)

In [264]:
fig = new_fig()

# Ajouter les points à la figure
add_points(points, fig, 'magenta')

# Ajouter le centre à la figure
add_point(center, fig, 'cyan')

fig.show()

### Obtenir les voisins

In [265]:
def get_direct_neighbors(points, num_neighbors):
    """
    Trouver les voisins directs de chaque point.
    Paramètres :
    - points (ndarray) : Un tableau de forme (num_points, 3) contenant les points en 3D.
    - num_neighbors (int) : Le nombre de voisins à trouver.
    Renvoie :
    - indices (ndarray) : Un tableau de forme (num_points, num_neighbors) contenant les indices des voisins directs.
    """
    nbrs = NearestNeighbors(n_neighbors=num_neighbors + 1, algorithm='kd_tree').fit(points)
    distances, indices = nbrs.kneighbors(points)
    return indices

direct_neighbors = get_direct_neighbors(points=points, num_neighbors=NUM_NEIGHBORS)

In [266]:
fig = new_fig()

# Ajouter les voisins directs à la figure
for i_point, point in enumerate(points):
    for i_neighbor in direct_neighbors[i_point]:
        neighbor = points[i_neighbor]
        add_line(point, neighbor, fig, 'blue')
        
fig.show()

### Obtenir les k plus proches voisins

In [267]:
def get_k_near_neighbors(direct_neighbors, k):
    """
    Trouver les k-voisins de chaque point.
    Paramètres :
    - direct_neighbors (ndarray) : Un tableau de forme (num_points, num_neighbors) contenant les indices des voisins directs.
    - k (int) : La profondeur de la recherche.
    Renvoie :
    - k_near_neighbors (list) : Une liste de longueur num_points contenant les indices des k-voisins de chaque point.
    """
    k_near_neighbors = [set() for _ in range(NUM_POINTS)]
    
    for i_point in range(NUM_POINTS):
        k_near_neighbors[i_point].add(i_point)

    for _ in range(k):
        for i_point in range(NUM_POINTS):
            for i_neighbor in direct_neighbors[i_point]:
                k_near_neighbors[i_point].update(direct_neighbors[i_neighbor])
    
    for i_point in range(NUM_POINTS):
        k_near_neighbors[i_point].remove(i_point)
    
    return k_near_neighbors


k_near_neighbors = get_k_near_neighbors(direct_neighbors=direct_neighbors, k=k)

In [268]:
fig = new_fig()

# Ajouter les points à la figure
add_points(points, fig, 'magenta')

# Ajouter un point d'example à la figure
example = np.random.randint(0, NUM_POINTS)
add_point(points[example], fig, 'red')

# Ajouter les k-voisins de l'exemple à la figure
for neighbor in k_near_neighbors[example]:
    add_point(points[neighbor], fig, 'orange')

to_plot_neighbors = [example]
for _ in range(k):
    new_neighbors = []
    for point in to_plot_neighbors:
        for neighbor in direct_neighbors[point]:
            new_neighbors.append(neighbor)
            add_line(points[point], points[neighbor], fig, 'blue')
    to_plot_neighbors = new_neighbors


fig.show()

### Calculer les normales sortantes

In [269]:
def get_normal(points, neighbors):
    """
    Trouver les vecteurs normaux de chaque point.
    Paramètres :
    - points (ndarray) : Un tableau de forme (num_points, 3) contenant les points en 3D.
    - neighbors (list) : Une liste de longueur num_points contenant les indices des k-voisins de chaque point.
    Renvoie :
    - normals (list) : Une liste de longueur num_points contenant les vecteurs normaux de chaque point.
    """
    T1 = []
    T2 = []
    normals = []
    
    for p in range(NUM_POINTS):
        neighbor_points = points[list(neighbors[p])]
        
        # Faire l'ACP
        pca = PCA(n_components=3)
        pca.fit(neighbor_points)
        
        T1.append(pca.components_[0])
        T2.append(pca.components_[1])

        normal_vector = pca.components_[2]
        if np.linalg.norm(points[p] - center + normal_vector) < np.linalg.norm(points[p] - center) :
            normal_vector =  - normal_vector
        normals.append(normal_vector)
    
    return normals, T1, T2


normals, T1, T2 = get_normal(points=points, neighbors=k_near_neighbors)

In [270]:
fig = new_fig()

# Ajouter les points à la figure
add_points(points, fig, 'magenta')

# Ajouter les points à la figure
add_point(center, fig, 'cyan')

# Ajouter les vecteurs normaux à la figure
for point, normal in zip(points, normals):
    add_line(point, point + normal, fig, 'orange')

fig.show()

### Trouver les rayons

In [271]:
def get_rayons(points, direct_neighbors):
    """
    Trouver les rayons de chaque point.
    Paramètres :
    - points (ndarray) : Un tableau de forme (num_points, 3) contenant les points en 3D.
    - direct_neighbors (ndarray) : Un tableau de forme (num_points, num_neighbors) contenant les indices des voisins directs.
    Renvoie :
    - rayons (list) : Une liste de longueur num_points contenant les rayons de chaque point.
    """
    rayons = []
    for i, voisins in enumerate(direct_neighbors):
        distances = [np.linalg.norm(points[i] - points[j]) for j in voisins]
        mean_distance = np.mean(distances)
        rayons.append(mean_distance / 2.0)
    return rayons

rayons = get_rayons(points=points, direct_neighbors=direct_neighbors)

In [272]:
fig = new_fig()

# Ajouter les points à la figure
for point, rayon, T1_point, T2_point in zip(points, rayons, T1, T2):
    add_circle(point, rayon, T1_point, T2_point, fig, 'lightgreen')

fig.show()

### Estimer la surface

In [273]:
# Créer un nuage de points
point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(points)

# Ajouter les normales au nuage de points
point_cloud.normals = o3d.utility.Vector3dVector(normals)
point_cloud.orient_normals_consistent_tangent_plane(50)

# Reconstruction de la surface avec la méthode de Poisson
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(point_cloud, depth=10, width=1)

vertices = np.asarray(mesh.vertices)
faces = np.asarray(mesh.triangles)

print("Nombre de points : ", len(vertices))
print("Nombre de faces : ", len(faces))

Nombre de points :  773
Nombre de faces :  1428


In [274]:
fig = new_fig()

# Ajouter la surface à la figure
fig.add_trace(go.Mesh3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    color='lightgreen'
))

fig.show()

### Trouver un point sur la surface

In [275]:
def found_Z(X, Y, tolerance):
    """
    Trouver la coordonnée Z d'un point (X, Y) sur la surface.
    Paramètres :
    - X (float) : La coordonnée X du point.
    - Y (float) : La coordonnée Y du point.
    - tolerance (float) : La tolérance pour la recherche du point.
    Renvoie :
    - Z (float) : La coordonnée Z du point trouvé.
    """
    p = [X, Y, 0]
    pas = 1

    dist = distance_point_surface(p, vertices, faces)
    while dist > tolerance:

        # On ajoute le pas
        p_plus = [p[0], p[1], p[2] + pas]
        dist_plus = distance_point_surface(p_plus, vertices, faces)

        # On enlève le pas
        p_moins = [p[0], p[1], p[2] - pas]
        dist_moins = distance_point_surface(p_moins, vertices, faces)

        # On regarde si un des deux est plus proche
        if dist_plus < dist:
            p = p_plus
            dist = dist_plus
        elif dist_moins < dist:
            p = p_moins
            dist = dist_moins
        else:
            pas /= 2

    return p[2]


X, Y = 2, 3
point_found = [X, Y, found_Z(X, Y, 1e-5)]

print("Point trouvé : ", point_found)

Point trouvé :  [2, 3, -1.0333404541015625]


In [276]:
fig = new_fig()

add_point(point_found, fig, 'blue')

# Ajouter la surface à la figure
fig.add_trace(go.Mesh3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    color='lightgreen'
))

fig.show()